In [1]:

pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os 
os.chdir('../')

In [3]:
import langchain
print(langchain.__version__)
import torch 
print(torch.__version__)


0.3.26
2.5.1


In [4]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter 
from langchain_community.document_loaders import PyMuPDFLoader


c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def load_pdf_files(data):
    loader=DirectoryLoader(data,
                           glob="*.pdf",
                           loader_cls=PyPDFLoader )
    documents =loader.load()
    return documents

In [21]:
extracted_data=load_pdf_files("data")

In [7]:
extracted_data[0:5]

[Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book (1).pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book (1).pdf', 'total_pages': 637, 'page': 1, 'page_label': '2'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book (1).pdf', 'total_pages': 637, 'page': 2, 'page_label': '3'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\

In [8]:
len(extracted_data)

637

In [6]:
from typing import List
from langchain.schema import Document


In [7]:
def filter_text(docs: List[Document]) -> List[Document]:
    return [
        Document(
            page_content=doc.page_content,
            metadata={"source": doc.metadata.get("source")}
        )
        for doc in docs
    ]


In [11]:
minimal_docs=filter_text(extracted_data)

In [12]:
minimal_docs[0:5]

[Document(metadata={'source': 'data\\Medical_book (1).pdf'}, page_content=''),
 Document(metadata={'source': 'data\\Medical_book (1).pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'data\\Medical_book (1).pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': 'data\\Medical_book (1).pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow,Manager, Imaging and Multimedia\nContent\nRobyn V . Young,Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imagi

In [8]:
def text_split(data):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    texts_chunks=text_splitter.split_documents(data)
    return texts_chunks

In [9]:
text_chunks=text_split(minimal_docs)
print(len(text_chunks))

NameError: name 'minimal_docs' is not defined

In [20]:
pip install --upgrade transformers

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.0 MB 882.6 kB/s eta 0:00:13
   -- ------------------------------------- 0.8/12.0 MB 882.6 kB/s eta 0:00:13
   --- ------------------------------------ 1.0/12.0 MB 898.8 kB/s eta 0:00:13
   --- ------------------------------------ 1.0/12.0 MB 898.8 kB/s eta 0:00:13
   ---- ----------------------------------- 1.3/12.0 MB 882.6 kB/s eta 0:00:13
   ----- ---------------------------------- 1.6/12.0 MB 882.6 kB/s eta 0:00:12
   ----- ---------------------------------- 1.6/12.0 MB 882.6 kB/s eta 0:00:12
   ------ --------------------------------- 1.8/12.0 MB 882.6 kB/s eta 0:00:12
   ------ --------------------------------- 2.1/12.0 MB 882.6 kB/s eta 0:00:12
   ------

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [11]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")  # or "cuda"

# Embed a query
vector = model.encode("Hello world", normalize_embeddings=True)
print(vector[:10])


[ 0.0151961  -0.02257071  0.0085471  -0.07417059  0.00383642  0.00271354
 -0.03126793  0.04463401  0.04405522 -0.00787117]


In [12]:
def download_emb():
    model_name = "BAAI/bge-small-en-v1.5"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cuda'},
        encode_kwargs={"normalize_embeddings": True}
    )
    return embeddings


In [13]:

import torch

# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available! GPU can be used.")
    print("GPU device name:", torch.cuda.get_device_name(0))
else:
    print("CUDA not available. Using CPU.")


CUDA is available! GPU can be used.
GPU device name: NVIDIA GeForce RTX 4050 Laptop GPU


In [15]:
embeddings=download_emb()

C:\Users\khali\AppData\Local\Temp\ipykernel_68368\1167181203.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [16]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='BAAI/bge-small-en-v1.5', cache_folder=None, model_kwargs={'device': 'cuda'}, encode_kwargs={'normalize_embeddings': True}, multi_process=False, show_progress=False)

In [21]:
vec=embeddings.embed_query("sup yssdfSfSfefdsfassine")

In [22]:
len(vec)

384

In [23]:
from dotenv import load_dotenv
import os
load_dotenv()


python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 12
python-dotenv could not parse statement starting at line 13
python-dotenv could not parse statement starting at line 14
python-dotenv could not parse statement starting at line 15
python-dotenv could not parse statement starting at line 16
python-dotenv could not parse statement starting at line 17


True

In [24]:
PINECONE_API_KEY=os.getenv("pinecone")
os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY

In [25]:
from pinecone import Pinecone 
pincone_api_key=PINECONE_API_KEY
pc=Pinecone(api_key=pincone_api_key)

In [26]:
pc

In [27]:
from pinecone import ServerlessSpec
index_name = "medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        index_name,
        dimension=384,
        metric="cosine",  
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index=pc.Index(index_name)

In [30]:
from langchain_pinecone import PineconeVectorStore


docsearch=PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

NameError: name 'text_chunks' is not defined

In [31]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    embedding=embeddings,   # your embeddings object
    index_name=index_name   # name of your existing Pinecone index
)


In [51]:
query = "cold feet"

results = docsearch.similarity_search(
    query=query,
    k=5
)


In [52]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)


• avoid tight shoes (especially in summer)
• wear sandals during warm weather
• wear cotton socks and change them often if they get
damp
• don’t wear socks made of synthetic material
• go barefoot outdoors when possible
• wear bathing shoes in public bathing or showering
areas
• use a good quality foot powder
• don’t wear sneakers without socks
• wash towels, contaminated floors, and shower stalls
well with hot soapy water if anyone in the family has
athlete’s foot
Resources
BOOKS
{'source': 'data\\Medical_book (1).pdf'}
hands and/or feet and a lack of pain. Cooling the hands
increases the blueness, while warming the hands decreas-
GALE ENCYCLOPEDIA OF MEDICINE 2 31
Acrocyanosis
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 31
{'source': 'data\\Medical_book (1).pdf'}
caused by constriction of the blood vessels in the
extremities, and occurs when the hands and feet
are exposed to cold weather. Emotional stress can
also trigger the cold symptoms.
Schizophrenia—Schizophrenia is a psychot